In [36]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/India_Runs_Hackathon', exist_ok=True)

print("Success! Your drive is mounted and the folder is ready.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Success! Your drive is mounted and the folder is ready.


In [37]:
!pip install -q sentence-transformers pandas numpy scikit-learn
print("AI dependencies successfully installed!")


AI dependencies successfully installed!


In [38]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading semantic search AI (takes 10-15 seconds)...")
ai_model = SentenceTransformer('all-MiniLM-L6-v2')

job_requirement = "Looking for a Data Scientist expert in Python, building ML pipelines, and predictive models."

names = ['Rahul Sharma', 'Amit Patel', 'Sneha Kulkarni', 'Vikram Singh']
exp = [5, 1, 4, 8]
resumes = ["Backend software engineer focused on Java, Spring Boot, and SQL databases.", "Junior developer with baseline knowledge of Python scripting and web scraping.", "Machine Learning Specialist expert in training neural networks, predictive analysis, and feature engineering packages.", "Senior Project Manager handling enterprise agile delivery, client communications, and budget planning."]

df = pd.DataFrame({'name': names, 'experience_years': exp, 'resume_text': resumes})

print("Analyzing candidates...")
job_vector = ai_model.encode([job_requirement])
candidate_vectors = ai_model.encode(df['resume_text'].tolist())

scores = cosine_similarity(job_vector, candidate_vectors).flatten()
df['match_score'] = scores

ranked_df = df.sort_values(by='match_score', ascending=False).reset_index(drop=True)
ranked_df['final_rank'] = ranked_df.index + 1

print("\n🏆 --- AI CANDIDATE RANKINGS --- 🏆")
print(ranked_df[['final_rank', 'name', 'match_score']])


Loading semantic search AI (takes 10-15 seconds)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Analyzing candidates...

🏆 --- AI CANDIDATE RANKINGS --- 🏆
   final_rank            name  match_score
0           1  Sneha Kulkarni     0.579198
1           2      Amit Patel     0.504889
2           3    Rahul Sharma     0.292739
3           4    Vikram Singh     0.182337


In [39]:
!unzip /content/drive/MyDrive/India_Runs_Hackathon/*.zip -d /content/drive/MyDrive/India_Runs_Hackathon/


unzip:  cannot find or open /content/drive/MyDrive/India_Runs_Hackathon/*.zip, /content/drive/MyDrive/India_Runs_Hackathon/*.zip.zip or /content/drive/MyDrive/India_Runs_Hackathon/*.zip.ZIP.

No zipfiles found.


In [40]:
!ls /content/drive/MyDrive/India_Runs_Hackathon/


'Idea Submission Template | Redrob_copy.pdf'


In [41]:
!ls "/content/drive/MyDrive/India_Runs_Hackathon/[PUB] India_runs_data_and_ai_challenge"


ls: cannot access '/content/drive/MyDrive/India_Runs_Hackathon/[PUB] India_runs_data_and_ai_challenge': No such file or directory


In [42]:
!ls "/content/drive/MyDrive/India_Runs_Hackathon/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge"


ls: cannot access '/content/drive/MyDrive/India_Runs_Hackathon/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge': No such file or directory


In [43]:
import pandas as pd
path = "/content/drive/MyDrive/India_Runs_Hackathon/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/"
print("Path shortcut set successfully!")


Path shortcut set successfully!


In [44]:
submission_df = pd.read_csv(path + "sample_submission.csv")
print("--- EXACT SUBMISSION COLUMNS REQUIRED ---")
print(submission_df.columns.tolist())
print("\n--- PREVIEW OF SUBMISSION FORMAT ---")
print(submission_df.head(3))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/India_Runs_Hackathon/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/sample_submission.csv'

In [ ]:
candidates_df = pd.read_json(path + "candidates.jsonl", lines=True)
print("--- REAL CANDIDATE DATA COLUMNS ---")
print(candidates_df.columns.tolist())
print("\n--- FIRST CANDIDATE PREVIEW ---")
print(candidates_df.iloc[0])


In [ ]:
!pip install -q docx2txt
print("Word document parser loaded!")


In [ ]:
import docx2txt
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

print("Step 1: Extracting target requirements from job_description.docx...")
job_desc_text = docx2txt.process(path + "job_description.docx")

print("Step 2: Processing complex nested data patterns from candidates.jsonl...")
# Parse nested fields into clean text streams without using indentations
candidates_df['headline'] = candidates_df['profile'].apply(lambda x: x.get('headline', '') if isinstance(x, dict) else '')
candidates_df['skills_text'] = candidates_df['skills'].apply(lambda x: ', '.join([s.get('name', '') for s in x if isinstance(s, dict)]) if isinstance(x, list) else '')

# Combine all profile attributes into a single searchable background description
candidates_df['search_profile'] = "Role Focus: " + candidates_df['headline'] + " | Competencies: " + candidates_df['skills_text']

print("Step 3: Generating mathematical match vectors via AI brain...")
job_vector = ai_model.encode([job_desc_text])
candidate_vectors = ai_model.encode(candidates_df['search_profile'].tolist(), show_progress_bar=True)

print("Step 4: Scoring and prioritizing candidates...")
raw_scores = cosine_similarity(job_vector, candidate_vectors).flatten()
candidates_df['score'] = raw_scores

# Sort immediately by score to find the best talent matches
final_submission = candidates_df.sort_values(by='score', ascending=False).reset_index(drop=True)
final_submission['rank'] = final_submission.index + 1

print("Step 5: Synthesizing the custom 'reasoning' descriptions required by judges...")
# Dynamically create the explanation text column using profile signals
final_submission['reasoning'] = "Candidate background emphasizes " + final_submission['headline'] + " possessing practical skills in: " + final_submission['skills_text'].str[:120] + "..."

print("Step 6: Cleaning columns to match requested schema specification...")
# Select ONLY the 4 exact columns requested in sample_submission.csv
submission_columns = ['candidate_id', 'rank', 'score', 'reasoning']
final_output_table = final_submission[submission_columns]

print("Step 7: Writing your finalized submission file straight to Google Drive...")
final_output_table.to_csv(path + "my_final_submission.csv", index=False)

print("\n🎉 CONGRATULATIONS! Pipeline Complete. Previewing top candidates:")
print(final_output_table.head(5))
